<a href="https://colab.research.google.com/github/sila85/production_ml_demo/blob/main/NLP35.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers datasets accelerate scikit-learn

In [ ]:
import os
import numpy as np
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

print("All libraries imported successfully!")
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

In [ ]:
imdb = load_dataset("stanfordnlp/imdb")

print(imdb)

In [ ]:
print("Movie Review:")
print(imdb["train"][0]["text"])

print("\nSentiment Label:")
print(imdb["train"][0]["label"])

In [ ]:
model_checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    model_checkpoint
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=2
)

print("Tokenizer loaded successfully!")
print("DistilBERT model loaded successfully!")

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )


tokenized_dataset = imdb.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# Rename label column for Hugging Face Trainer
tokenized_dataset = tokenized_dataset.rename_column(
    "label", "labels"
)

print("Tokenization completed successfully!")

print(tokenized_dataset["train"][0])

In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Dynamic padding configured successfully!")

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)

    precision = precision_score(
        labels,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        labels,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        labels,
        predictions,
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


print("Evaluation metrics defined successfully!")

In [ ]:
training_args = TrainingArguments(
    output_dir="./distilbert_imdb_results",

    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_strategy="steps",
    logging_steps=100,

    save_total_limit=2,

    fp16=torch.cuda.is_available(),

    report_to="none",

    seed=42
)

print("Training configuration completed!")

In [ ]:
import inspect

trainer_parameters = inspect.signature(
    Trainer.__init__
).parameters

trainer_options = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_dataset["train"],
    "eval_dataset": tokenized_dataset["test"],
    "data_collator": data_collator,
    "compute_metrics": compute_metrics
}

# Support both newer and older Transformers versions
if "processing_class" in trainer_parameters:
    trainer_options["processing_class"] = tokenizer
elif "tokenizer" in trainer_parameters:
    trainer_options["tokenizer"] = tokenizer

trainer = Trainer(**trainer_options)

print("Trainer created successfully!")

In [ ]:
train_results = trainer.train()

print("Training completed successfully!")

print(train_results)

In [ ]:
# Split the original training data into 90% training and 10% validation
train_validation = tokenized_dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

train_data = train_validation["train"]
validation_data = train_validation["test"]

print("Training samples:", len(train_data))
print("Validation samples:", len(validation_data))
print("Final test samples:", len(tokenized_dataset["test"]))

In [ ]:
trainer_options["train_dataset"] = train_data
trainer_options["eval_dataset"] = validation_data

trainer = Trainer(**trainer_options)

print("Trainer updated with training and validation datasets!")

In [ ]:
test_results = trainer.evaluate(
    eval_dataset=tokenized_dataset["test"]
)

print("\nFINAL TEST RESULTS")

for metric, value in test_results.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
prediction_output = trainer.predict(
    tokenized_dataset["test"]
)

predicted_labels = np.argmax(
    prediction_output.predictions,
    axis=1
)

actual_labels = prediction_output.label_ids

print("\nCLASSIFICATION REPORT\n")

print(
    classification_report(
        actual_labels,
        predicted_labels,
        labels=[0, 1],
        target_names=["Negative", "Positive"],
        zero_division=0
    )
)

In [ ]:
cm = confusion_matrix(
    actual_labels,
    predicted_labels,
    labels=[0, 1]
)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Positive"]
)

display.plot(values_format="d")

plt.title("DistilBERT IMDb Sentiment Classification")
plt.show()

print("Confusion Matrix:")
print(cm)

In [ ]:
def predict_sentiment(review):

    model.eval()

    inputs = tokenizer(
        review,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    # Move inputs to the same device as the model
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model(**inputs)

        probabilities = torch.softmax(
            outputs.logits,
            dim=1
        )

    predicted_class = torch.argmax(
        probabilities,
        dim=1
    ).item()

    sentiment = (
        "Positive" if predicted_class == 1
        else "Negative"
    )

    confidence = probabilities[0, predicted_class].item()

    return sentiment, confidence

In [ ]:
review = "This movie was fantastic. The acting was excellent and the storyline was amazing."

sentiment, confidence = predict_sentiment(review)

print("Review:", review)
print("Predicted sentiment:", sentiment)
print("Confidence:", round(confidence * 100, 2), "%")

In [ ]:
review = "This movie was terrible. The acting was poor and the storyline was boring."

sentiment, confidence = predict_sentiment(review)

print("Review:", review)
print("Predicted sentiment:", sentiment)
print("Confidence:", round(confidence * 100, 2), "%")

In [ ]:
save_directory = "./final_distilbert_imdb"

trainer.save_model(save_directory)

tokenizer.save_pretrained(save_directory)

print("Model and tokenizer saved successfully!")

In [ ]:
!zip -r final_distilbert_imdb.zip final_distilbert_imdb